In [ ]:
"""
KNOW-NET Baseline Models Implementation
All models use same preprocessed data and DBpedia embeddings as DEAP-FAKED paper
"""

# First install required packages
!pip install transformers datasets -q
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!python -m spacy download en_core_web_sm -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import pickle
import json
import os
import requests
import time
from tqdm import tqdm
import random
import re
import hashlib
from collections import defaultdict, Counter
import spacy
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score, confusion_matrix
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_extraction.text import CountVectorizer
import warnings
warnings.filterwarnings('ignore')
from transformers import RobertaModel, RobertaTokenizerFast
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
import gc

# Set seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================================
# CONFIGURATION
# ============================================================================

class BaselineConfig:
    # Model parameters
    vocab_size = 10000
    embedding_dim = 300
    hidden_dim = 256
    kg_embed_dim = 300  # DBpedia embeddings dimension

    # Training parameters
    learning_rate = 0.001
    batch_size = 32
    num_epochs = 10
    max_seq_length = 256
    max_entities = 5

    # File paths
    train_path = "train_deap.csv"
    val_path = "val_deap.csv"
    test_path = "test_deap.csv"
    dbpedia_embeddings_path = "dbpedia_embeddings.pkl"

    # Output
    results_dir = "baseline_models_results"

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = BaselineConfig()

# ============================================================================
# STEP 1: DATA PROCESSING FOR ALL MODELS
# ============================================================================

class TextDataset(Dataset):
    """Basic text dataset for models without entity information"""

    def __init__(self, df, vocab=None, max_length=256):
        self.df = df.reset_index(drop=True)
        self.max_length = max_length

        if vocab is None:
            # Build vocabulary from training data
            self.vocab = self._build_vocab()
        else:
            self.vocab = vocab

        # Precompute sequences
        self.sequences = self._precompute_sequences()

    def _build_vocab(self):
        """Build vocabulary from all texts"""
        word_counts = Counter()
        for text in self.df['title']:
            words = str(text).lower().split()
            word_counts.update(words)

        # Keep most common words
        most_common = word_counts.most_common(config.vocab_size - 2)
        vocab = {'<PAD>': 0, '<UNK>': 1}
        for idx, (word, _) in enumerate(most_common):
            vocab[word] = idx + 2

        return vocab

    def _precompute_sequences(self):
        """Convert texts to sequences of word indices"""
        sequences = []

        for text in self.df['title']:
            words = str(text).lower().split()
            seq = []
            for word in words[:self.max_length]:
                if word in self.vocab:
                    seq.append(self.vocab[word])
                else:
                    seq.append(1)  # UNK token

            # Pad if necessary
            if len(seq) < self.max_length:
                seq = seq + [0] * (self.max_length - len(seq))
            else:
                seq = seq[:self.max_length]

            sequences.append(seq)

        return torch.tensor(sequences, dtype=torch.long)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return {
            'sequence': self.sequences[idx],
            'label': torch.tensor(self.df.iloc[idx]['label'], dtype=torch.long),
            'text': str(self.df.iloc[idx]['title'])
        }

class EntityDataset(Dataset):
    """Dataset for models with entity information"""

    def __init__(self, df, embeddings_dict, vocab=None, max_length=256):
        self.df = df.reset_index(drop=True)
        self.embeddings_dict = embeddings_dict
        self.max_length = max_length

        # Initialize spaCy for entity extraction
        try:
            self.nlp = spacy.load("en_core_web_sm")
        except:
            import subprocess
            subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm", "-q"])
            self.nlp = spacy.load("en_core_web_sm")

        if vocab is None:
            self.vocab = self._build_vocab()
        else:
            self.vocab = vocab

        # Add unknown entity embedding
        if '<UNK_ENTITY>' not in self.embeddings_dict:
            rng = np.random.RandomState(999999)
            unk_embedding = rng.randn(config.kg_embed_dim).astype(np.float32)
            unk_embedding = unk_embedding / np.linalg.norm(unk_embedding)
            self.embeddings_dict['<UNK_ENTITY>'] = unk_embedding

        # Precompute data
        self.sequences, self.entity_embeddings = self._precompute_data()

    def _build_vocab(self):
        """Build vocabulary"""
        word_counts = Counter()
        for text in self.df['title']:
            words = str(text).lower().split()
            word_counts.update(words)

        most_common = word_counts.most_common(config.vocab_size - 2)
        vocab = {'<PAD>': 0, '<UNK>': 1}
        for idx, (word, _) in enumerate(most_common):
            vocab[word] = idx + 2

        return vocab

    def get_entity_embedding(self, entity_name):
        """Get embedding for an entity"""
        entity_name = str(entity_name).strip()

        if entity_name in self.embeddings_dict:
            return self.embeddings_dict[entity_name]

        for key in self.embeddings_dict.keys():
            if entity_name.lower() == key.lower():
                return self.embeddings_dict[key]

        return self.embeddings_dict['<UNK_ENTITY>']

    def extract_entities(self, text):
        """Extract entities from text"""
        doc = self.nlp(text)
        entities = []

        for ent in doc.ents:
            if ent.label_ in ['PERSON', 'ORG', 'GPE', 'LOC', 'NORP', 'EVENT', 'MISC']:
                entities.append(ent.text.strip())

        return entities[:config.max_entities]

    def _precompute_data(self):
        """Precompute sequences and entity embeddings"""
        sequences = []
        all_entity_embeddings = []

        for text in self.df['title']:
            # Convert text to sequence
            words = str(text).lower().split()
            seq = []
            for word in words[:self.max_length]:
                if word in self.vocab:
                    seq.append(self.vocab[word])
                else:
                    seq.append(1)

            if len(seq) < self.max_length:
                seq = seq + [0] * (self.max_length - len(seq))
            else:
                seq = seq[:self.max_length]

            sequences.append(seq)

            # Extract and embed entities
            entities = self.extract_entities(text)
            kg_embeddings = []
            for entity in entities:
                emb = self.get_entity_embedding(entity)
                kg_embeddings.append(emb)

            # Pad entity embeddings
            if kg_embeddings:
                kg_tensor = np.stack(kg_embeddings)
                if kg_tensor.shape[0] < config.max_entities:
                    padding = np.zeros((config.max_entities - kg_tensor.shape[0], config.kg_embed_dim))
                    kg_tensor = np.concatenate([kg_tensor, padding], axis=0)
            else:
                kg_tensor = np.zeros((config.max_entities, config.kg_embed_dim))

            all_entity_embeddings.append(kg_tensor)

        return torch.tensor(sequences, dtype=torch.long), torch.tensor(all_entity_embeddings, dtype=torch.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return {
            'sequence': self.sequences[idx],
            'entity_embeddings': self.entity_embeddings[idx],
            'label': torch.tensor(self.df.iloc[idx]['label'], dtype=torch.long),
            'text': str(self.df.iloc[idx]['title'])
        }

# ============================================================================
# STEP 2: BASELINE MODELS IMPLEMENTATION
# ============================================================================

# ==================== MODEL 1: ExtraTreeClassifier ====================

def run_extratree_classifier(train_df, val_df, test_df):
    """ExtraTreeClassifier baseline (Bag-of-words)"""
    print("\n" + "="*60)
    print("ExtraTreeClassifier Baseline")
    print("="*60)

    # Prepare text data
    train_texts = train_df['title'].astype(str).tolist()
    val_texts = val_df['title'].astype(str).tolist()
    test_texts = test_df['title'].astype(str).tolist()

    train_labels = train_df['label'].tolist()
    val_labels = val_df['label'].tolist()
    test_labels = test_df['label'].tolist()

    # Create bag-of-words features
    vectorizer = CountVectorizer(max_features=config.vocab_size)
    X_train = vectorizer.fit_transform(train_texts)
    X_val = vectorizer.transform(val_texts)
    X_test = vectorizer.transform(test_texts)

    # Train model
    print("Training ExtraTreeClassifier...")
    model = ExtraTreesClassifier(n_estimators=100, random_state=SEED)
    model.fit(X_train, train_labels)

    # Evaluate
    train_preds = model.predict(X_train)
    val_preds = model.predict(X_val)
    test_preds = model.predict(X_test)

    train_acc = accuracy_score(train_labels, train_preds)
    val_acc = accuracy_score(val_labels, val_preds)
    test_acc = accuracy_score(test_labels, test_preds)

    train_f1 = f1_score(train_labels, train_preds, average='macro')
    val_f1 = f1_score(val_labels, val_preds, average='macro')
    test_f1 = f1_score(test_labels, test_preds, average='macro')

    print(f"Train Accuracy: {train_acc:.4f}, F1: {train_f1:.4f}")
    print(f"Val Accuracy:   {val_acc:.4f}, F1: {val_f1:.4f}")
    print(f"Test Accuracy:  {test_acc:.4f}, F1: {test_f1:.4f}")

    results = {
        'model': 'ExtraTreeClassifier',
        'train_accuracy': train_acc,
        'train_f1': train_f1,
        'val_accuracy': val_acc,
        'val_f1': val_f1,
        'test_accuracy': test_acc,
        'test_f1': test_f1,
        'predictions': test_preds,
        'labels': test_labels
    }

    return results

# ==================== MODEL 2: LSTM ====================

class LSTMModel(nn.Module):
    """Vanilla LSTM model"""
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(LSTMModel, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=False
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, 2)
        )

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hidden, _) = self.lstm(embedded)
        features = hidden[-1]
        logits = self.classifier(features)
        return logits

def run_lstm(train_df, val_df, test_df):
    """LSTM baseline"""
    print("\n" + "="*60)
    print("LSTM Baseline")
    print("="*60)

    # Create datasets
    train_dataset = TextDataset(train_df)
    val_dataset = TextDataset(val_df, vocab=train_dataset.vocab)
    test_dataset = TextDataset(test_df, vocab=train_dataset.vocab)

    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

    # Create model
    model = LSTMModel(
        vocab_size=len(train_dataset.vocab),
        embedding_dim=config.embedding_dim,
        hidden_dim=config.hidden_dim
    )
    model.to(config.device)

    # Training setup
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

    # Training loop
    best_val_f1 = 0
    patience = 5
    patience_counter = 0

    for epoch in range(config.num_epochs):
        # Train
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
            sequences = batch['sequence'].to(config.device)
            labels = batch['label'].to(config.device)

            optimizer.zero_grad()
            outputs = model(sequences)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_acc = train_correct / train_total

        # Validate
        model.eval()
        val_preds = []
        val_labels_list = []

        with torch.no_grad():
            for batch in val_loader:
                sequences = batch['sequence'].to(config.device)
                labels = batch['label'].to(config.device)

                outputs = model(sequences)
                _, predicted = torch.max(outputs.data, 1)

                val_preds.extend(predicted.cpu().numpy())
                val_labels_list.extend(labels.cpu().numpy())

        val_acc = accuracy_score(val_labels_list, val_preds)
        val_f1 = f1_score(val_labels_list, val_preds, average='macro')

        print(f"Epoch {epoch+1}: Train Loss: {train_loss/len(train_loader):.4f}, Train Acc: {train_acc:.4f}")
        print(f"              Val Acc: {val_acc:.4f}, Val F1: {val_f1:.4f}")

        # Early stopping
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    # Load best model
    model.load_state_dict(best_model_state)

    # Test
    model.eval()
    test_preds = []
    test_labels_list = []

    with torch.no_grad():
        for batch in test_loader:
            sequences = batch['sequence'].to(config.device)
            labels = batch['label'].to(config.device)

            outputs = model(sequences)
            _, predicted = torch.max(outputs.data, 1)

            test_preds.extend(predicted.cpu().numpy())
            test_labels_list.extend(labels.cpu().numpy())

    test_acc = accuracy_score(test_labels_list, test_preds)
    test_f1 = f1_score(test_labels_list, test_preds, average='macro')

    print(f"\nTest Results: Accuracy: {test_acc:.4f}, F1: {test_f1:.4f}")

    results = {
        'model': 'LSTM',
        'test_accuracy': test_acc,
        'test_f1': test_f1,
        'best_val_f1': best_val_f1,
        'predictions': test_preds,
        'labels': test_labels_list
    }

    return results

# ==================== MODEL 3: SentRoBERTa ====================

class SentRoBERTaModel(nn.Module):
    """Sentence RoBERTa model"""
    def __init__(self):
        super(SentRoBERTaModel, self).__init__()
        self.roberta = RobertaModel.from_pretrained('roberta-base')
        self.classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 2)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_embedding)
        return logits

def run_sentroberta(train_df, val_df, test_df):
    """SentRoBERTa baseline"""
    print("\n" + "="*60)
    print("SentRoBERTa Baseline")
    print("="*60)

    # Initialize tokenizer
    tokenizer = RobertaTokenizerFast.from_pretrained('roberta-base')

    # Tokenize data
    def tokenize_data(df):
        texts = df['title'].astype(str).tolist()
        encodings = tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=config.max_seq_length,
            return_tensors='pt'
        )
        return encodings

    train_encodings = tokenize_data(train_df)
    val_encodings = tokenize_data(val_df)
    test_encodings = tokenize_data(test_df)

    # Create datasets
    class RobertaDataset(Dataset):
        def __init__(self, encodings, labels):
            self.encodings = encodings
            self.labels = labels

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, idx):
            return {
                'input_ids': self.encodings['input_ids'][idx],
                'attention_mask': self.encodings['attention_mask'][idx],
                'label': torch.tensor(self.labels[idx], dtype=torch.long)
            }

    train_dataset = RobertaDataset(train_encodings, train_df['label'].tolist())
    val_dataset = RobertaDataset(val_encodings, val_df['label'].tolist())
    test_dataset = RobertaDataset(test_encodings, test_df['label'].tolist())

    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

    # Create model
    model = SentRoBERTaModel()
    model.to(config.device)

    # Freeze some layers
    for param in model.roberta.encoder.layer[:6].parameters():
        param.requires_grad = False

    # Training setup
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

    # Training loop
    best_val_f1 = 0
    patience = 5
    patience_counter = 0

    for epoch in range(config.num_epochs):
        # Train
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
            input_ids = batch['input_ids'].to(config.device)
            attention_mask = batch['attention_mask'].to(config.device)
            labels = batch['label'].to(config.device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_acc = train_correct / train_total

        # Validate
        model.eval()
        val_preds = []
        val_labels_list = []

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(config.device)
                attention_mask = batch['attention_mask'].to(config.device)
                labels = batch['label'].to(config.device)

                outputs = model(input_ids, attention_mask)
                _, predicted = torch.max(outputs.data, 1)

                val_preds.extend(predicted.cpu().numpy())
                val_labels_list.extend(labels.cpu().numpy())

        val_acc = accuracy_score(val_labels_list, val_preds)
        val_f1 = f1_score(val_labels_list, val_preds, average='macro')

        print(f"Epoch {epoch+1}: Train Loss: {train_loss/len(train_loader):.4f}, Train Acc: {train_acc:.4f}")
        print(f"              Val Acc: {val_acc:.4f}, Val F1: {val_f1:.4f}")

        # Early stopping
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    # Load best model
    model.load_state_dict(best_model_state)

    # Test
    model.eval()
    test_preds = []
    test_labels_list = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(config.device)
            attention_mask = batch['attention_mask'].to(config.device)
            labels = batch['label'].to(config.device)

            outputs = model(input_ids, attention_mask)
            _, predicted = torch.max(outputs.data, 1)

            test_preds.extend(predicted.cpu().numpy())
            test_labels_list.extend(labels.cpu().numpy())

    test_acc = accuracy_score(test_labels_list, test_preds)
    test_f1 = f1_score(test_labels_list, test_preds, average='macro')

    print(f"\nTest Results: Accuracy: {test_acc:.4f}, F1: {test_f1:.4f}")

    results = {
        'model': 'SentRoBERTa',
        'test_accuracy': test_acc,
        'test_f1': test_f1,
        'best_val_f1': best_val_f1,
        'predictions': test_preds,
        'labels': test_labels_list
    }

    return results

# ==================== MODEL 4: StackedBiLSTM ====================

class StackedBiLSTMModel(nn.Module):
    """2-layer Stacked BiLSTM model (DEAP-FAKED's text encoder)"""
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(StackedBiLSTMModel, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.bilstm = nn.LSTM(
            embedding_dim,
            hidden_dim // 2,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=0.3
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, 2)
        )

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hidden, _) = self.bilstm(embedded)

        # Concatenate forward and backward hidden states
        hidden_forward = hidden[-2, :, :]
        hidden_backward = hidden[-1, :, :]
        features = torch.cat((hidden_forward, hidden_backward), dim=1)

        logits = self.classifier(features)
        return logits

def run_stacked_bilstm(train_df, val_df, test_df):
    """StackedBiLSTM baseline"""
    print("\n" + "="*60)
    print("StackedBiLSTM Baseline")
    print("="*60)

    # Create datasets
    train_dataset = TextDataset(train_df)
    val_dataset = TextDataset(val_df, vocab=train_dataset.vocab)
    test_dataset = TextDataset(test_df, vocab=train_dataset.vocab)

    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

    # Create model
    model = StackedBiLSTMModel(
        vocab_size=len(train_dataset.vocab),
        embedding_dim=config.embedding_dim,
        hidden_dim=config.hidden_dim
    )
    model.to(config.device)

    # Training setup
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

    # Training loop
    best_val_f1 = 0
    patience = 5
    patience_counter = 0

    for epoch in range(config.num_epochs):
        # Train
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
            sequences = batch['sequence'].to(config.device)
            labels = batch['label'].to(config.device)

            optimizer.zero_grad()
            outputs = model(sequences)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_acc = train_correct / train_total

        # Validate
        model.eval()
        val_preds = []
        val_labels_list = []

        with torch.no_grad():
            for batch in val_loader:
                sequences = batch['sequence'].to(config.device)
                labels = batch['label'].to(config.device)

                outputs = model(sequences)
                _, predicted = torch.max(outputs.data, 1)

                val_preds.extend(predicted.cpu().numpy())
                val_labels_list.extend(labels.cpu().numpy())

        val_acc = accuracy_score(val_labels_list, val_preds)
        val_f1 = f1_score(val_labels_list, val_preds, average='macro')

        print(f"Epoch {epoch+1}: Train Loss: {train_loss/len(train_loader):.4f}, Train Acc: {train_acc:.4f}")
        print(f"              Val Acc: {val_acc:.4f}, Val F1: {val_f1:.4f}")

        # Early stopping
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    # Load best model
    model.load_state_dict(best_model_state)

    # Test
    model.eval()
    test_preds = []
    test_labels_list = []

    with torch.no_grad():
        for batch in test_loader:
            sequences = batch['sequence'].to(config.device)
            labels = batch['label'].to(config.device)

            outputs = model(sequences)
            _, predicted = torch.max(outputs.data, 1)

            test_preds.extend(predicted.cpu().numpy())
            test_labels_list.extend(labels.cpu().numpy())

    test_acc = accuracy_score(test_labels_list, test_preds)
    test_f1 = f1_score(test_labels_list, test_preds, average='macro')

    print(f"\nTest Results: Accuracy: {test_acc:.4f}, F1: {test_f1:.4f}")

    results = {
        'model': 'StackedBiLSTM',
        'test_accuracy': test_acc,
        'test_f1': test_f1,
        'best_val_f1': best_val_f1,
        'predictions': test_preds,
        'labels': test_labels_list
    }

    return results

# ==================== MODEL 5: EntWiki-StackedBiLSTM ====================

class EntWikiStackedBiLSTMModel(nn.Module):
    """Entity+Wikipedia StackedBiLSTM model"""
    def __init__(self, vocab_size, embedding_dim, hidden_dim, kg_embed_dim):
        super(EntWikiStackedBiLSTMModel, self).__init__()

        # Text encoder (StackedBiLSTM)
        self.text_embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.text_encoder = nn.LSTM(
            embedding_dim,
            hidden_dim // 2,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=0.3
        )

        # Entity encoder (using DBpedia embeddings)
        self.entity_encoder = nn.Linear(kg_embed_dim, hidden_dim)

        # Classifier
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, text_seq, entity_embeddings):
        # Text encoding
        text_embedded = self.text_embedding(text_seq)
        text_out, (text_hidden, _) = self.text_encoder(text_embedded)
        text_features = torch.cat((text_hidden[-2], text_hidden[-1]), dim=1)

        # Entity encoding (mean pooling of all entity embeddings)
        entity_features = self.entity_encoder(entity_embeddings)
        entity_features = torch.mean(entity_features, dim=1)  # Mean pooling

        # Combine features
        combined_features = torch.cat([text_features, entity_features], dim=1)
        logits = self.classifier(combined_features)

        return logits

def run_entwiki_stacked_bilstm(train_df, val_df, test_df, embeddings_dict):
    """EntWiki-StackedBiLSTM baseline"""
    print("\n" + "="*60)
    print("EntWiki-StackedBiLSTM Baseline")
    print("="*60)

    # Create datasets
    train_dataset = EntityDataset(train_df, embeddings_dict)
    val_dataset = EntityDataset(val_df, embeddings_dict, vocab=train_dataset.vocab)
    test_dataset = EntityDataset(test_df, embeddings_dict, vocab=train_dataset.vocab)

    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

    # Create model
    model = EntWikiStackedBiLSTMModel(
        vocab_size=len(train_dataset.vocab),
        embedding_dim=config.embedding_dim,
        hidden_dim=config.hidden_dim,
        kg_embed_dim=config.kg_embed_dim
    )
    model.to(config.device)

    # Training setup
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

    # Training loop
    best_val_f1 = 0
    patience = 5
    patience_counter = 0

    for epoch in range(config.num_epochs):
        # Train
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
            sequences = batch['sequence'].to(config.device)
            entity_embeddings = batch['entity_embeddings'].to(config.device)
            labels = batch['label'].to(config.device)

            optimizer.zero_grad()
            outputs = model(sequences, entity_embeddings)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_acc = train_correct / train_total

        # Validate
        model.eval()
        val_preds = []
        val_labels_list = []

        with torch.no_grad():
            for batch in val_loader:
                sequences = batch['sequence'].to(config.device)
                entity_embeddings = batch['entity_embeddings'].to(config.device)
                labels = batch['label'].to(config.device)

                outputs = model(sequences, entity_embeddings)
                _, predicted = torch.max(outputs.data, 1)

                val_preds.extend(predicted.cpu().numpy())
                val_labels_list.extend(labels.cpu().numpy())

        val_acc = accuracy_score(val_labels_list, val_preds)
        val_f1 = f1_score(val_labels_list, val_preds, average='macro')

        print(f"Epoch {epoch+1}: Train Loss: {train_loss/len(train_loader):.4f}, Train Acc: {train_acc:.4f}")
        print(f"              Val Acc: {val_acc:.4f}, Val F1: {val_f1:.4f}")

        # Early stopping
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    # Load best model
    model.load_state_dict(best_model_state)

    # Test
    model.eval()
    test_preds = []
    test_labels_list = []

    with torch.no_grad():
        for batch in test_loader:
            sequences = batch['sequence'].to(config.device)
            entity_embeddings = batch['entity_embeddings'].to(config.device)
            labels = batch['label'].to(config.device)

            outputs = model(sequences, entity_embeddings)
            _, predicted = torch.max(outputs.data, 1)

            test_preds.extend(predicted.cpu().numpy())
            test_labels_list.extend(labels.cpu().numpy())

    test_acc = accuracy_score(test_labels_list, test_preds)
    test_f1 = f1_score(test_labels_list, test_preds, average='macro')

    print(f"\nTest Results: Accuracy: {test_acc:.4f}, F1: {test_f1:.4f}")

    results = {
        'model': 'EntWiki-StackedBiLSTM',
        'test_accuracy': test_acc,
        'test_f1': test_f1,
        'best_val_f1': best_val_f1,
        'predictions': test_preds,
        'labels': test_labels_list
    }

    return results

# ============================================================================
# STEP 3: MAIN EXECUTION
# ============================================================================

def setup_directories():
    """Create directories"""
    os.makedirs(config.results_dir, exist_ok=True)
    print(f"Results directory: {config.results_dir}")

def load_data():
    """Load preprocessed data"""
    print("\n1. LOADING DATA")
    print("-" * 40)

    train_df = pd.read_csv(config.train_path)
    val_df = pd.read_csv(config.val_path)
    test_df = pd.read_csv(config.test_path)

    print(f"Train: {len(train_df):,} samples")
    print(f"Val:   {len(val_df):,} samples")
    print(f"Test:  {len(test_df):,} samples")

    return train_df, val_df, test_df

def load_embeddings():
    """Load DBpedia embeddings"""
    print("\n2. LOADING EMBEDDINGS")
    print("-" * 40)

    if not os.path.exists(config.dbpedia_embeddings_path):
        print(f"Error: {config.dbpedia_embeddings_path} not found!")
        return None

    with open(config.dbpedia_embeddings_path, 'rb') as f:
        embeddings = pickle.load(f)

    print(f"Loaded {len(embeddings):,} DBpedia embeddings")
    return embeddings

def run_all_baselines():
    """Run all baseline models"""
    print("="*80)
    print("RUNNING ALL BASELINE MODELS FROM DEAP-FAKED PAPER")
    print("="*80)

    # Setup
    setup_directories()

    # Load data
    train_df, val_df, test_df = load_data()

    # Load embeddings
    embeddings_dict = load_embeddings()

    all_results = []

    # Run each baseline model
    print("\n" + "="*80)
    print("RUNNING BASELINE MODELS")
    print("="*80)

    # 1. ExtraTreeClassifier
    try:
        print("\n[1/5] Running ExtraTreeClassifier...")
        results_et = run_extratree_classifier(train_df, val_df, test_df)
        all_results.append(results_et)
    except Exception as e:
        print(f"Error in ExtraTreeClassifier: {e}")

    # 2. LSTM
    try:
        print("\n[2/5] Running LSTM...")
        results_lstm = run_lstm(train_df, val_df, test_df)
        all_results.append(results_lstm)
    except Exception as e:
        print(f"Error in LSTM: {e}")

    # 3. SentRoBERTa
    try:
        print("\n[3/5] Running SentRoBERTa...")
        results_sr = run_sentroberta(train_df, val_df, test_df)
        all_results.append(results_sr)
    except Exception as e:
        print(f"Error in SentRoBERTa: {e}")

    # 4. StackedBiLSTM
    try:
        print("\n[4/5] Running StackedBiLSTM...")
        results_sblstm = run_stacked_bilstm(train_df, val_df, test_df)
        all_results.append(results_sblstm)
    except Exception as e:
        print(f"Error in StackedBiLSTM: {e}")

    # 5. EntWiki-StackedBiLSTM (only if embeddings available)
    if embeddings_dict is not None:
        try:
            print("\n[5/5] Running EntWiki-StackedBiLSTM...")
            results_entwiki = run_entwiki_stacked_bilstm(train_df, val_df, test_df, embeddings_dict)
            all_results.append(results_entwiki)
        except Exception as e:
            print(f"Error in EntWiki-StackedBiLSTM: {e}")

    # Save all results
    print("\n" + "="*80)
    print("SAVING RESULTS")
    print("="*80)

    # Create results DataFrame
    results_df = pd.DataFrame([
        {
            'model': r['model'],
            'test_accuracy': r.get('test_accuracy', 0),
            'test_f1': r.get('test_f1', 0),
            'best_val_f1': r.get('best_val_f1', 0)
        }
        for r in all_results
    ])

    results_csv_path = f"{config.results_dir}/baseline_results.csv"
    results_df.to_csv(results_csv_path, index=False)
    print(f"Results saved to: {results_csv_path}")

    # Save detailed predictions for each model
    for results in all_results:
        if 'predictions' in results and 'labels' in results:
            preds_df = pd.DataFrame({
                'true_label': results['labels'],
                'predicted_label': results['predictions']
            })
            model_name = results['model'].replace('-', '_').replace(' ', '_').lower()
            preds_csv_path = f"{config.results_dir}/{model_name}_predictions.csv"
            preds_df.to_csv(preds_csv_path, index=False)
            print(f"  {results['model']} predictions saved")

    # Print comparison table
    print("\n" + "="*80)
    print("BASELINE MODELS COMPARISON")
    print("="*80)
    print("\nPerformance Summary:")
    print("-" * 60)
    print(f"{'Model':<25} {'Test Accuracy':>15} {'Test F1':>15}")
    print(f"{'-'*25} {'-'*15} {'-'*15}")

    for results in all_results:
        model_name = results['model']
        test_acc = results.get('test_accuracy', 0)
        test_f1 = results.get('test_f1', 0)
        print(f"{model_name:<25} {test_acc:>15.4f} {test_f1:>15.4f}")

    return all_results

def compare_with_know_net(baseline_results, know_net_results_path="know_net_accurate_ner_results/results.csv"):
    """Compare baseline models with KNOW-NET results"""
    print("\n" + "="*80)
    print("COMPARISON WITH KNOW-NET")
    print("="*80)

    # Load KNOW-NET results if available
    if os.path.exists(know_net_results_path):
        know_net_df = pd.read_csv(know_net_results_path)
        kn_acc = know_net_df['test_accuracy'].values[0]
        kn_f1 = know_net_df['test_f1_score'].values[0]

        print(f"\nKNOW-NET Results: Accuracy = {kn_acc:.4f}, F1 = {kn_f1:.4f}")

        # Create comparison table
        print(f"\n{'Model':<25} {'Test Accuracy':>15} {'Test F1':>15} {'vs KNOW-NET (F1)':>20}")
        print(f"{'-'*25} {'-'*15} {'-'*15} {'-'*20}")

        # Add baseline results
        for results in baseline_results:
            model_name = results['model']
            test_acc = results.get('test_accuracy', 0)
            test_f1 = results.get('test_f1', 0)
            diff_f1 = test_f1 - kn_f1
            diff_str = f"{diff_f1:+.4f}"

            print(f"{model_name:<25} {test_acc:>15.4f} {test_f1:>15.4f} {diff_str:>20}")

        # Add KNOW-NET at the end
        print(f"{'-'*25} {'-'*15} {'-'*15} {'-'*20}")
        print(f"{'KNOW-NET (Accurate NER)':<25} {kn_acc:>15.4f} {kn_f1:>15.4f} {'(baseline)':>20}")

        # Statistical insights
        print(f"\nKey Insights:")
        print(f"  1. KNOW-NET uses cross-attention while baselines use concatenation or no fusion")
        print(f"  2. KNOW-NET uses RoBERTa while most baselines use LSTM/BiLSTM")
        print(f"  3. KNOW-NET includes NER auxiliary task for regularization")
        print(f"  4. All models use same data and DBpedia embeddings")

    else:
        print("KNOW-NET results not found for comparison!")
        print(f"Expected file: {know_net_results_path}")

# ============================================================================
# EXECUTE
# ============================================================================

if __name__ == "__main__":
    # Check for required files
    required_files = [
        config.train_path,
        config.val_path,
        config.test_path
    ]

    missing = [f for f in required_files if not os.path.exists(f)]

    if missing:
        print("Missing required files:")
        for f in missing:
            print(f"  - {f}")
        print("\nPlease ensure DEAP-FAKED preprocessing was completed.")
    else:
        print("All required files found!")
        print("\n" + "="*80)
        print("STARTING BASELINE MODELS IMPLEMENTATION")
        print("="*80)

        try:
            baseline_results = run_all_baselines()
            if baseline_results:
                compare_with_know_net(baseline_results)
                print("\n" + "="*80)
                print("BASELINE MODELS IMPLEMENTATION COMPLETE")
                print("="*80)
        except Exception as e:
            print(f"\nError: {e}")
            import traceback
            traceback.print_exc()

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
All required files found!

STARTING BASELINE MODELS IMPLEMENTATION
RUNNING ALL BASELINE MODELS FROM DEAP-FAKED PAPER
Results directory: baseline_models_results

1. LOADING DATA
----------------------------------------
Train: 20,953 samples
Val:   2,619 samples
Test:  2,620 samples

2. LOADING EMBEDDINGS
----------------------------------------
Loaded 2,786 DBpedia embeddings

RUNNING BASELINE MODELS

[1/5] Running ExtraTreeClassifier...

ExtraTreeClassifier Baseline
Training ExtraTreeClassifier...
Train Accuracy: 1.0000, F1: 1.0000
Val Accuracy:   0.9557, F1: 0.9557
Test Accuracy:  0.9611, F1: 0.9611

[2/5] Running LSTM...

LSTM Baseline


Epoch 1: Train Loss: 0.6933, Train Acc: 0.5014
              Val Acc: 0.5078, Val F1: 0.3368


Epoch 2: Train Loss: 0.6932, Train Acc: 0.5069
              Val Acc: 0.5078, Val F1: 0.3368


Epoch 3: Train Loss: 0.6932, Train Acc: 0.5079
              Val Acc: 0.5078, Val F1: 0.3368


Epoch 4: Train Loss: 0.6931, Train Acc: 0.5059
              Val Acc: 0.5078, Val F1: 0.3368


...

Epoch 6: Train Loss: 0.0079, Train Acc: 0.9973
              Val Acc: 0.9603, Val F1: 0.9603


Epoch 7: Train Loss: 0.0080, Train Acc: 0.9977
              Val Acc: 0.9622, Val F1: 0.9622


Epoch 8: Train Loss: 0.0065, Train Acc: 0.9980
              Val Acc: 0.9572, Val F1: 0.9572


Epoch 9: Train Loss: 0.0080, Train Acc: 0.9979
              Val Acc: 0.9580, Val F1: 0.9580


Epoch 10: Train Loss: 0.0056, Train Acc: 0.9983
              Val Acc: 0.9599, Val F1: 0.9599
Early stopping at epoch 10

Test Results: Accuracy: 0.9714, F1: 0.9714

SAVING RESULTS
Results saved to: baseline_models_results/baseline_results.csv
  ExtraTreeClassifier predictions saved
  LSTM predictions saved
  SentRoBERTa predictions saved
  StackedBiLSTM predictions saved
  EntWiki-StackedBiLSTM predictions saved

BASELINE MODELS COMPARISON

Performance Summary:
------------------------------------------------------------
Model                       Test Accuracy         Test F1
------------------------- --------------- ---------------
ExtraTreeClassifier                0.9611          0.9611
LSTM                               0.5080          0.3369
SentRoBERTa                        0.9847          0.9847
StackedBiLSTM                      0.9660          0.9660
EntWiki-StackedBiLSTM              0.9714          0.9714

COMPARISON WITH KNOW-NET
KNOW-NET results not found for compariso